# 11_01_上限の見積もりと実測

コード：`ch11/blocks/11_01_上限の見積もりと実測.ipynb`

本文のコードをコードセルで直接実行します。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 11 章：メモリ帯域とモデルのバイト数から decode 速度の上限を見積もり、実測と比べる
import time, numpy as np, torch

# ① 見積もり表：tok/s の上限 ≒ 帯域 ÷ 1 トークンで読むバイト数（≒ モデルのバイト数）
print("パラメータ数 × ビット数 → バイト数、帯域ごとの上限 tok/s")
for params in (1e9, 3e9, 8e9):
    for bits in (16, 8, 4):
        nbytes = params * bits / 8
        row = "  ".join(f"{bw:>3d}GB/s:{bw * 1e9 / nbytes:6.1f}" for bw in (50, 100, 200, 400))
        print(f"  {params / 1e9:.0f}B {bits:2d}bit {nbytes / 1e9:5.1f}GB  {row}")

# ② この機械の実効メモリ帯域（大きな配列のコピー：読み＋書き）
a = np.ones(512 * 1024 * 1024 // 8)                 # 512 MiB
b = np.empty_like(a)
best = 0
for _ in range(5):
    t0 = time.perf_counter(); np.copyto(b, a); dt = time.perf_counter() - t0
    best = max(best, 2 * a.nbytes / dt / 1e9)
print(f"② 実効メモリ帯域（コピー）：{best:.1f} GB/s")

# ③ 実モデルの decode 速度（CPU・fp32 と bf16）
from transformers import AutoModelForCausalLM, AutoTokenizer
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
ids = tok("日本の首都は", return_tensors="pt").input_ids
torch.set_num_threads(torch.get_num_threads())
for dtype in (torch.float32, torch.bfloat16):
    model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).eval()
    nbytes = sum(p.numel() * p.element_size() for p in model.parameters())
    with torch.no_grad():
        model.generate(ids, max_new_tokens=4, do_sample=False)             # 準備運転
        t0 = time.perf_counter()
        out = model.generate(ids, max_new_tokens=64, min_new_tokens=64, do_sample=False)
        dt = time.perf_counter() - t0
    n = out.shape[1] - ids.shape[1]
    print(f"③ {str(dtype).split('.')[-1]:8s} 重み {nbytes / 1e9:.2f} GB  実測 {n / dt:5.1f} tok/s  "
          f"帯域からの上限 {best * 1e9 / nbytes:5.1f} tok/s  （上限の {n / dt / (best * 1e9 / nbytes):.0%}）")
